# Fase 2 — Modelado experimental de intenciones PQRS

Este cuaderno ejecuta el flujo implementado en `src/pqrs_nlp/` y los scripts del repositorio, sin mantener una implementación paralela. El corpus es sintético; las expresiones `wayuunaiki_basico` son preliminares y no han sido validadas por hablantes competentes.

Objetivo: comparar una referencia Dummy, TF-IDF + regresión logística y TF-IDF de palabras/caracteres + LinearSVC. La métrica principal es F1 macro. Las particiones y los pliegues separan grupos `template_id` para reducir la fuga por plantillas.

## 1. Preparación reproducible y control de calidad

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
subprocess.run([sys.executable, 'scripts/prepare_dataset.py'], cwd=ROOT, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

dataset = pd.read_csv(ROOT / 'data/processed/pqrs_sinteticas_limpias.csv')
quality = json.loads((ROOT / 'data/processed/data_quality.json').read_text(encoding='utf-8'))
display(pd.Series(quality, name='control de calidad'))
display(dataset.head())
display(dataset.groupby('intencion').size().rename('registros'))

## 2. Normalización y particiones

Se normalizan mayúsculas y espacios, pero se preservan tildes, negaciones y puntuación. `intencion` es el objetivo; identificadores, nivel de lenguaje, entidades y metadatos de generación no entran como características.

In [ ]:
from pqrs_nlp.data import split_by_template
splits, split_metadata = split_by_template(dataset, seed=42)
split_metadata

## 3. Baseline, modelo propuesto, búsqueda y validación

El pipeline fija los transformadores dentro de cada pliegue. GridSearchCV prueba `C = [0.1, 1, 10]` y `class_weight = [None, 'balanced']` con F1 macro y validación cruzada agrupada. La evaluación reservada de prueba se ejecuta una sola vez después de seleccionar por validación.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_experiments.py'], cwd=ROOT, check=True)
validation = pd.read_csv(ROOT / 'reports/metrics/validation_metrics.csv')
tuning = pd.read_csv(ROOT / 'reports/metrics/tuning_results.csv')
test = json.loads((ROOT / 'reports/metrics/test_metrics.json').read_text(encoding='utf-8'))
display(validation)
display(tuning)
display(pd.Series({k: test[k] for k in ('selected_model', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted')}))

## 4. Evaluación por nivel, matriz de confusión y errores

El análisis por `nivel_lenguaje` se realiza después de predecir y no usa dicho metadato como entrada. Las cifras de grupos pequeños son inestables. La categoría wayuunaiki es solo experimental; no demuestra atención fiable en esa lengua.

In [ ]:
display(pd.read_csv(ROOT / 'reports/metrics/test_by_language.csv'))
display(pd.read_csv(ROOT / 'reports/metrics/test_by_language_per_class.csv'))
display(pd.read_csv(ROOT / 'reports/metrics/test_per_class.csv'))
display(pd.read_csv(ROOT / 'reports/metrics/test_errors.csv'))
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports/figures/test_confusion_matrix.png')))

## 5. Conclusiones y trabajo futuro

Las conclusiones se generan a partir de `reports/metrics/test_metrics.json`, no de resultados escritos de antemano. Debido a la naturaleza sintética y repetitiva del corpus, una puntuación alta no implica generalización a usuarios reales. Se requiere validación externa con mensajes independientes y revisión lingüística competente de cualquier contenido wayuunaiki antes de valorar nuevas aplicaciones.